# Skin Lesion Robustness -- Quick Training Run

Trains baseline vs augmented MobileNetV2 on 500 images, 2 epochs. ~2 min on GPU.

In [ ]:
!pip install -q datasets scikit-learn

In [ ]:
import os, io, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [ ]:
print('Loading data (streaming, grabs only what we need)...')
ds = load_dataset('ahmed-ai/skin-lesions-classification-dataset', split='train', streaming=True)

images, labels = [], []
for i, row in enumerate(ds):
    if i >= 800: break
    images.append(row['image'].convert('RGB'))
    labels.append(str(row['label']))

label_names = sorted(set(labels))
NC = len(label_names)
l2i = {l:i for i,l in enumerate(label_names)}
i2l = {i:l for l,i in l2i.items()}
int_labels = [l2i[l] for l in labels]

# shuffle before splitting so test set isn't all one class
combined = list(zip(images, int_labels))
random.shuffle(combined)
images, int_labels = zip(*combined)
images, int_labels = list(images), list(int_labels)

train_imgs, train_lbls = images[:600], int_labels[:600]
test_imgs, test_lbls = images[600:], int_labels[600:]
print(f'Train: {len(train_imgs)}, Test: {len(test_imgs)}, Classes: {NC}')
print(f'Class names: {label_names}')

In [ ]:
def shift_tone(img, f):
    a = np.asarray(img, dtype=np.float32)
    lin = np.power(np.clip(a/255,0,1), 2.2) * f
    return Image.fromarray((np.clip(np.power(lin,1/2.2),0,1)*255).astype(np.uint8))

def warm(img, s=.15):
    a=np.asarray(img,dtype=np.float32); a[...,0]=np.clip(a[...,0]*(1+s),0,255); a[...,2]=np.clip(a[...,2]*(1-s),0,255)
    return Image.fromarray(a.astype(np.uint8))

def cool(img, s=.15):
    a=np.asarray(img,dtype=np.float32); a[...,0]=np.clip(a[...,0]*(1-s),0,255); a[...,2]=np.clip(a[...,2]*(1+s),0,255)
    return Image.fromarray(a.astype(np.uint8))

def noise(img, sigma=15):
    a=np.asarray(img,dtype=np.float32)
    return Image.fromarray(np.clip(a+np.random.normal(0,sigma,a.shape),0,255).astype(np.uint8))

def jpegq(img, q=25):
    b=io.BytesIO(); img.save(b,format='JPEG',quality=q); b.seek(0); return Image.open(b).copy()

PERTS = {
    'darker_skin':    lambda im: shift_tone(im, 0.55),
    'lighter_skin':   lambda im: shift_tone(im, 1.45),
    'low_light':      lambda im: ImageEnhance.Brightness(im).enhance(0.55),
    'harsh_light':    lambda im: ImageEnhance.Brightness(ImageEnhance.Contrast(im).enhance(1.4)).enhance(1.25),
    'warm_white_bal': lambda im: warm(im, 0.2),
    'cool_white_bal': lambda im: cool(im, 0.2),
    'motion_blur':    lambda im: im.filter(ImageFilter.BoxBlur(5)),
    'out_of_focus':   lambda im: im.filter(ImageFilter.GaussianBlur(2.5)),
    'sensor_noise':   lambda im: noise(im, 18),
    'jpeg_artifacts': lambda im: jpegq(im, 20),
    'off_axis':       lambda im: im.rotate(15, resample=Image.BILINEAR, fillcolor=(0,0,0)),
}
print(f'{len(PERTS)} perturbations')

In [ ]:
norm = transforms.Normalize([.485,.456,.406],[.229,.224,.225])
base_tfm = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
test_tfm = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), norm])

class AugTfm:
    def __init__(self):
        self.post = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])
    def __call__(self, img):
        if random.random()<0.5:
            try: img = random.choice(list(PERTS.values()))(img)
            except: pass
        if random.random()<0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
        return self.post(img)

class ListDS(Dataset):
    def __init__(self, imgs, lbls, tfm):
        self.imgs, self.lbls, self.tfm = imgs, lbls, tfm
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tfm(self.imgs[i]), self.lbls[i]

bl_loader = DataLoader(ListDS(train_imgs, train_lbls, base_tfm), batch_size=32, shuffle=True)
aug_loader = DataLoader(ListDS(train_imgs, train_lbls, AugTfm()), batch_size=32, shuffle=True)
tst_loader = DataLoader(ListDS(test_imgs, test_lbls, test_tfm), batch_size=32)
print('Loaders ready')

In [ ]:
def make():
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.features.parameters(): p.requires_grad=False
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.last_channel, NC))
    return m.to(device)

def run_train(model, loader, ep=2):
    crit=nn.CrossEntropyLoss(); opt=optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=1e-3)
    hist={'loss':[],'acc':[]}
    for e in range(ep):
        model.train(); rl=c=t=0; t0=time.time()
        for x,y in loader:
            x,y=x.to(device),y.to(device); opt.zero_grad(); o=model(x); l=crit(o,y); l.backward(); opt.step()
            rl+=l.item()*x.size(0); c+=(o.argmax(1)==y).sum().item(); t+=y.size(0)
        hist['loss'].append(rl/t); hist['acc'].append(c/t)
        print(f'  Ep {e+1}/{ep} loss:{rl/t:.4f} acc:{c/t:.4f} ({time.time()-t0:.1f}s)')
    return hist

print('--- BASELINE ---')
bl_m = make(); bl_h = run_train(bl_m, bl_loader, 2)
print('\n--- AUGMENTED ---')
aug_m = make(); aug_h = run_train(aug_m, aug_loader, 2)

In [ ]:
@torch.no_grad()
def ev(model, loader):
    model.eval(); ps,ls=[],[]
    for x,y in loader:
        ps.extend(model(x.to(device)).argmax(1).cpu().numpy()); ls.extend(y.numpy())
    return np.array(ls), np.array(ps)

bl_l, bl_p = ev(bl_m, tst_loader)
aug_l, aug_p = ev(aug_m, tst_loader)
bl_acc = accuracy_score(bl_l, bl_p)
aug_acc = accuracy_score(aug_l, aug_p)
print(f'Clean test -- Baseline: {bl_acc:.4f}, Augmented: {aug_acc:.4f}')

present = sorted(set(bl_l) | set(bl_p) | set(aug_p))
tgt_names = [i2l.get(i, str(i)) for i in present]
print('\nBaseline:'); print(classification_report(bl_l, bl_p, labels=present, target_names=tgt_names, zero_division=0))
print('Augmented:'); print(classification_report(aug_l, aug_p, labels=present, target_names=tgt_names, zero_division=0))

In [ ]:
class PertDS(Dataset):
    def __init__(self, imgs, lbls, pfn, tfm):
        self.imgs,self.lbls,self.pfn,self.tfm = imgs,lbls,pfn,tfm
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tfm(self.pfn(self.imgs[i])), self.lbls[i]

res = {}
for pn, pf in PERTS.items():
    pl = DataLoader(PertDS(test_imgs, test_lbls, pf, test_tfm), batch_size=32)
    _, bpp = ev(bl_m, pl); _, app = ev(aug_m, pl)
    res[pn] = {'bl_acc':accuracy_score(bl_l,bpp), 'aug_acc':accuracy_score(aug_l,app),
               'bl_agree':np.mean(bpp==bl_p), 'aug_agree':np.mean(app==aug_p)}
    print(f'{pn:<18} BL:{res[pn]["bl_acc"]:.3f} AUG:{res[pn]["aug_acc"]:.3f} BL_agr:{res[pn]["bl_agree"]:.3f} AUG_agr:{res[pn]["aug_agree"]:.3f}')

In [ ]:
names=list(res.keys()); x=np.arange(len(names)); w=0.38
fig,(a1,a2)=plt.subplots(2,1,figsize=(12,9))

a1.bar(x-w/2,[res[n]['bl_acc'] for n in names],w,label='Baseline',color='#e74c3c')
a1.bar(x+w/2,[res[n]['aug_acc'] for n in names],w,label='Augmented',color='#2ecc71')
a1.axhline(bl_acc,color='#e74c3c',ls=':',alpha=.5); a1.axhline(aug_acc,color='#2ecc71',ls=':',alpha=.5)
a1.set_ylabel('Accuracy'); a1.set_title('Accuracy under perturbation'); a1.set_xticks(x)
a1.set_xticklabels(names,rotation=35,ha='right'); a1.legend(fontsize=9); a1.grid(axis='y',alpha=.3); a1.set_ylim(0,1.05)

a2.bar(x-w/2,[res[n]['bl_agree'] for n in names],w,label='Baseline',color='#e74c3c')
a2.bar(x+w/2,[res[n]['aug_agree'] for n in names],w,label='Augmented',color='#2ecc71')
a2.axhline(1,color='gray',ls=':',lw=.8)
a2.set_ylabel('Agreement with clean pred'); a2.set_title('Prediction stability (higher = more robust)'); a2.set_xticks(x)
a2.set_xticklabels(names,rotation=35,ha='right'); a2.legend(fontsize=9); a2.grid(axis='y',alpha=.3); a2.set_ylim(0,1.05)

fig.tight_layout(); plt.savefig('robustness_comparison.png',dpi=150); plt.show()

In [ ]:
sample = test_imgs[0]
fig,axes=plt.subplots(3,4,figsize=(16,10)); axes=axes.flatten()
axes[0].imshow(sample); axes[0].set_title('Original',fontweight='bold'); axes[0].axis('off')
for i,(nm,fn) in enumerate(PERTS.items()):
    axes[i+1].imshow(fn(sample)); axes[i+1].set_title(nm,fontsize=10); axes[i+1].axis('off')
fig.suptitle('All 11 perturbations',fontsize=14); fig.tight_layout()
plt.savefig('perturbation_gallery.png',dpi=150); plt.show()

In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
cm1=confusion_matrix(bl_l,bl_p); cm2=confusion_matrix(aug_l,aug_p)
a1.imshow(cm1,cmap='Blues'); a1.set_title(f'Baseline ({bl_acc:.3f})'); a1.set_xlabel('Pred'); a1.set_ylabel('Actual')
a2.imshow(cm2,cmap='Greens'); a2.set_title(f'Augmented ({aug_acc:.3f})'); a2.set_xlabel('Pred'); a2.set_ylabel('Actual')
for i in range(NC):
    for j in range(NC):
        a1.text(j,i,str(cm1[i][j]),ha='center',va='center',fontsize=8)
        a2.text(j,i,str(cm2[i][j]),ha='center',va='center',fontsize=8)
fig.tight_layout(); plt.savefig('confusion_matrices.png',dpi=150); plt.show()
print('Done! Download PNGs from sidebar.')